In [1]:
from pathlib import Path

from tracers.project import setup_project
from tracers.datasets.factory import build_datasets, build_loader_for_model
from tracers.pipelines.trainer_factory import build_trainer

PROJECT_ROOT = Path.cwd().parent
setup = setup_project(PROJECT_ROOT)

MODEL_KEY = "tracer_np"

experiment_config = setup["experiment_config"]
hyperparams = setup["hyperparams"]
segment_frames = setup["segment_frames"]
batch_size = setup["batch_size"]
device = setup["device"]

In [2]:
audio_dir, beatmap_dir, train_dataset, val_dataset, test_dataset, train_ids, val_ids, test_ids = build_datasets(
    PROJECT_ROOT,
    dataset_folder_name="dataset-augment",
    segment_frames=segment_frames,
    seed=42,
)

train_loader = build_loader_for_model(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    model_key=MODEL_KEY,
)
val_loader = build_loader_for_model(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)
test_loader = build_loader_for_model(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    model_key=MODEL_KEY,
)

trainer, model = build_trainer(
    model_key=MODEL_KEY,
    hyperparams=hyperparams,
    experiment_config=experiment_config,
    device=device,
    checkpoint_path=(
        str(PROJECT_ROOT / experiment_config.resume_from)
        if experiment_config.resume_from is not None
        else None
    ),
    train_loader=train_loader,
    valid_loader=val_loader,
)

if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from

    # Load pretrained weights, but NOT the old Adam optimizer state.
    trainer.load_checkpoint(
        resume_from,
        load_optimizer=False,
    )

    # Set the fine-tuning learning rate.
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    # Start a new experiment.
    trainer.start_epoch = 0
    trainer.best_val_loss = float("inf")

    print("Fine-tuning from:", resume_from)
    print("Learning rate:", hyperparams["learning_rate"])
    print("Optimizer state: reset")

Loaded (epoch 5, best val loss 0.1488)
Loaded (epoch 5, best val loss 0.1488)
Fine-tuning from: c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt
Learning rate: 5e-05
Optimizer state: reset


In [3]:
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\joshua\_netrc.
wandb: Currently logged in as: joshuardh (joshuardh-mindanao-state-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Validation: 100%|██████████| 151/151 [00:25<00:00,  5.96it/s]



Epoch 1/20

TRAIN
Loss   | Total: 0.1152 | NP: 0.0087 | NS: 0.0546
Onset  | P: 0.8419 | R: 0.8493 | F1: 0.8456
Action | P: 0.9158 | R: 0.8926 | F1: 0.9040
Active Exact Accuracy: 0.4160
GT Active Rate: 5.00% | Predicted Active Rate: 4.87%

VAL
Loss   | Total: 0.1549 | NP: 0.0137 | NS: 0.0590
Onset  | P: 0.7583 | R: 0.7415 | F1: 0.7498
Action | P: 0.8650 | R: 0.8443 | F1: 0.8545
Active Exact Accuracy: 0.3784
GT Active Rate: 4.37% | Predicted Active Rate: 4.26%


Epochs:   5%|▌         | 1/20 [09:26<2:59:27, 566.72s/it]

Saved checkpoint: checkpoints\best.pt
Saved checkpoint: checkpoints\latest.pt


Epochs:  10%|█         | 2/20 [18:20<2:44:09, 547.18s/it]

... (34 outputs left out)


Epoch 19/20

TRAIN
Loss   | Total: 0.0939 | NP: 0.0069 | NS: 0.0455
Onset  | P: 0.8720 | R: 0.8831 | F1: 0.8775
Action | P: 0.9310 | R: 0.9188 | F1: 0.9249
Active Exact Accuracy: 0.4584
GT Active Rate: 5.00% | Predicted Active Rate: 4.93%

VAL
Loss   | Total: 0.1667 | NP: 0.0153 | NS: 0.0597
Onset  | P: 0.7208 | R: 0.7792 | F1: 0.7489
Action | P: 0.8373 | R: 0.8788 | F1: 0.8576
Active Exact Accuracy: 0.4026
GT Active Rate: 4.37% | Predicted Active Rate: 4.59%
Saved checkpoint: checkpoints\latest.pt


Epochs: 100%|██████████| 20/20 [2:58:16<00:00, 534.84s/it]



Epoch 20/20

TRAIN
Loss   | Total: 0.0929 | NP: 0.0069 | NS: 0.0449
Onset  | P: 0.8730 | R: 0.8839 | F1: 0.8785
Action | P: 0.9321 | R: 0.9192 | F1: 0.9256
Active Exact Accuracy: 0.4601
GT Active Rate: 4.99% | Predicted Active Rate: 4.92%

VAL
Loss   | Total: 0.1713 | NP: 0.0157 | NS: 0.0612
Onset  | P: 0.7221 | R: 0.7755 | F1: 0.7479
Action | P: 0.8301 | R: 0.8791 | F1: 0.8539
Active Exact Accuracy: 0.3994
GT Active Rate: 4.37% | Predicted Active Rate: 4.63%
Saved checkpoint: checkpoints\latest.pt


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
time/epoch_seconds,█▂▂▂▂▂▂▂▂▂▁▂▃▂▂▂▂▇▂▁
time/train_seconds,█▂▂▂▂▂▂▂▂▂▂▂▃▂▂▂▂▇▂▁
time/val_seconds,█▂▂▂▂▃▂▂▂▂▁▇▂▂▃▂▁█▃▁
train/action_f1,▁▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇▇▇██
train/action_precision,▁▂▂▃▃▃▃▄▄▄▅▅▆▆▆▇▇▇██
train/action_recall,▁▂▂▃▃▃▄▄▄▅▅▆▆▇▇▇▇▇██
train/active_exact_accuracy,▁▂▂▂▃▃▄▄▄▅▅▆▆▆▆▇▇███
train/gt_active_rate,▆█▆▇▇▆▆▅▄▄▃▆▅▆▅▁▅▃▆▅
train/np_loss,██▇▆▆▆▅▅▅▄▄▃▃▃▃▂▂▂▁▁
+18,...
